# LightRAG (2024)
---
[[paper]](https://arxiv.org/pdf/2402.13872)<br>LightRAG = Lightweight Retrieval-Augmented Generation

LightRAG — это фреймворк, предложенный исследователями из Университета Цинхуа и других институтов, который фокусируется на создании эффективных и экономичных систем Retrieval-Augmented Generation (RAG).

### Проблема

Традиционные системы RAG, такие как Vanilla RAG (2020), показали значительные улучшения в задачах генерации, но их широкое внедрение сдерживается несколькими факторами:
1.  **Высокие вычислительные затраты:** Зависимость от больших языковых моделей (LLM) и часто мощных Dense Retrieval моделей требует значительных ресурсов для обучения и инференса.
2.  **Задержки (Latency):** Обработка больших контекстов и выполнение запросов к крупным LLM приводит к высоким задержкам, что критично для приложений реального времени.
3.  **Стоимость:** Использование больших моделей, особенно проприетарных API, влечет за собой существенные операционные расходы.
4.  **Сложность управления:** Поддержание и масштабирование больших RAG-систем может быть трудоемким.

### Идея

Основная идея LightRAG заключается в создании **легковесного и экономичного RAG-фреймворка** путем оптимизации каждого его компонента и введения новых механизмов, которые компенсируют ограничения небольших моделей. Это достигается за счет:
1.  Использования **компактных, специально настроенных LLM** в качестве генератора.
2.  Применения **легковесных ретриверов**.
3.  Внедрения продвинутых стратегий **уточнения промптов (prompt refinement)** и **самокоррекции** для максимизации эффективности небольших моделей.

### Постановка задачи

Фреймворк LightRAG решает задачу повышения эффективности, снижения стоимости и уменьшения задержек систем RAG при сохранении конкурентоспособного качества генерации ответов на запросы, например, в задачах Question Answering или суммаризации.

### Существующие альтернативы

На момент появления LightRAG существовали различные подходы к RAG, каждый со своими особенностями:
*   **Vanilla RAG (2020) и REALM (2020):** Первые полноценные RAG-модели, использующие большие трансформерные LLM (например, T5, BERT) и обученные на сквозных задачах. Их основной недостаток — высокая ресурсоемкость.
*   **RETRO (2021):** Модель, которая интегрирует механизм Retrieval непосредственно в архитектуру LLM во время предварительного обучения, но также требует использования крупномасштабных языковых моделей.
*   **Гибридные ретриверы:** Некоторые системы комбинировали лексические методы (например, BM25) с Dense Retrieval для улучшения качества, но это добавляло сложности и иногда не снижало общие затраты на инференс.
*   **Дистилляция LLM:** Были попытки дистиллировать большие LLM в меньшие, но часто это приводило к потере способности эффективно обрабатывать сложные запросы и контексты, особенно в RAG-сценариях.

Основное архитектурное отличие LightRAG заключается в его **комплексном подходе к легковесности** на уровне всего фреймворка, а не только отдельного компонента. В отличие от предыдущих работ, LightRAG предлагает не просто уменьшить размер LLM, а **активно компенсировать недостатки маленькой LLM** с помощью интеллектуальных механизмов ретривала и промптинга.

### Архитектура

Архитектура LightRAG состоит из четырех ключевых модулей, работающих сообща для обеспечения легковесной и эффективной генерации:

1.  **Lightweight Generator (Генератор):**
    *   Представляет собой небольшую языковую модель (например, GPT-2 Small, Llama-2-7B, Phi-2), которая специально дообучена для выполнения RAG-задач. Выбор небольшой модели значительно снижает вычислительные затраты и задержки по сравнению с крупными LLM.
2.  **Lightweight Retriever (Ретривер):**
    *   Отвечает за поиск релевантных документов или отрывков из базы знаний. Он может быть реализован как:
        *   **Лексический ретривер:** Например, BM25, который очень быстр и экономичен.
        *   **Легковесный Dense Retriever:** Небольшая модель-кодировщик (например, мини-BERT), обученная для создания плотных эмбеддингов, что позволяет быстрее и дешевле индексировать и искать по сравнению с большими BERT-подобными моделями.
    *   Поддерживает **многоуровневый ретривер (multi-granularity retrieval)**: возможность извлекать информацию на разных уровнях детализации (например, предложения, абзацы, полные документы), что позволяет более гибко подходить к формированию контекста.
3.  **Cross-Modality Prompt Refinement (CMPR) Module (Модуль уточнения промпта):**
    *   Это ключевой компонент, который берет извлеченные ретривером фрагменты и исходный запрос, а затем **оптимизирует и конденсирует** их в более релевантный и краткий промпт для Генератора.
    *   Модуль CMPR действует как "интеллектуальный фильтр и суммаризатор", который компенсирует ограниченную способность небольших LLM обрабатывать длинные и шумные контексты. Он может использовать техники ранжирования предложений, экстрактивного или абстрактивного суммаризации, чтобы сформировать наиболее полезный контекст.
4.  **Self-Correction Mechanism (SCM) (Механизм самокоррекции):**
    *   Этот механизм контролирует процесс генерации ответов Генератором. Если во время генерации обнаруживаются потенциальные ошибки, несоответствия или недостаточность информации, SCM может:
        *   **Инициировать повторный ретривал:** Сформулировать новый, более уточненный подзапрос для Ретривера.
        *   **Инициировать повторную генерацию:** Повторно запустить Генератор с измененным или расширенным контекстом/промптом.
    *   SCM действует как "монитор и перепланировщик", который позволяет маленькой LLM выполнять итеративный процесс проверки и уточнения, приближая качество к большим моделям.

### Алгоритм обучения

LightRAG — это фреймворк, поэтому обучение относится к его отдельным компонентам:

1.  **Обучение Генератора:**
    *   Компактная LLM дообучается (fine-tuning) или настраивается (instruction tuning) на наборах данных, специфичных для RAG (например, пары "запрос-контекст-ответ"). Цель — научить модель генерировать релевантные ответы на основе предоставленного контекста.
2.  **Обучение Ретривера (если используется Dense Retriever):**
    *   Небольшая модель-кодировщик обучается с использованием методов контрастивного обучения (contrastive learning) на наборах данных, содержащих релевантные и нерелевантные пары "запрос-документ". Это аналогично обучению таких моделей, как DPR.
3.  **Обучение CMPR (если это модель):**
    *   Если модуль CMPR реализован как отдельная модель (например, суммаризатор), он может обучаться на данных, где требуется извлечь или суммировать ключевую информацию из длинных текстов, чтобы получить более короткий и информативный промпт. В простейших случаях это может быть и эвристический модуль.
4.  **SCM:** Механизм самокоррекции чаще всего реализуется через правила или небольшой классификатор, который оценивает качество сгенерированного ответа и принимает решение о необходимости итерации. Его "обучение" может сводиться к настройке порогов или обучению классификатора на примерах "хороших" и "плохих" ответов.

### Алгоритм инференса

1.  **Вход:** Пользовательский запрос $Q$.
2.  **Шаг 1: Исходный ретривал:**
    *   Ретривер (Lightweight Retriever) использует запрос $Q$ для поиска $K$ наиболее релевантных фрагментов (документов, абзацев, предложений) из базы знаний. В этом шаге может применяться многоуровневый ретривал.
3.  **Шаг 2: Уточнение промпта:**
    *   Извлеченные фрагменты вместе с исходным запросом $Q$ передаются в модуль CMPR. CMPR обрабатывает эту информацию, чтобы создать **оптимизированный, компактный и наиболее релевантный контекст (промпт)** для Генератора.
4.  **Шаг 3: Исходная генерация:**
    *   Оптимизированный промпт подается на вход Генератору (Lightweight Generator), который генерирует первый черновик ответа.
5.  **Шаг 4: Самокоррекция:**
    *   Механизм SCM анализирует сгенерированный ответ на предмет качества, релевантности и полноты.
    *   Если SCM обнаруживает проблемы (например, ответ неполный, противоречивый, не соответствует запросу), он может принять решение:
        *   **Повторить ретривал:** Сформировать новый подзапрос или изменить стратегию ретривала (например, искать более детальную информацию).
        *   **Повторить генерацию:** Передать Генератору улучшенный промпт (например, с дополнительной информацией или инструкциями) для повторной генерации.
    *   Этот итеративный цикл продолжается до тех пор, пока не будет получен удовлетворительный ответ или не будет достигнуто максимальное число итераций.
6.  **Шаг 5: Вывод:** Финальный сгенерированный ответ.

### Результаты

LightRAG демонстрирует значительные улучшения в эффективности и стоимости при сохранении конкурентоспособного качества:

*   На таких бенчмарках, как Natural Questions и HotpotQA, LightRAG с гораздо меньшими моделями (например, Llama-2-7B) достигает производительности, сопоставимой или даже превосходящей более крупные RAG-системы (например, RAG на базе Llama-2-13B) с точки зрения точности ответов.
*   Фреймворк демонстрирует существенное снижение вычислительных затрат (FLOPs, потребление памяти) и задержек инференса. Например, LightRAG на Llama-2-7B может обеспечить 2-3-кратное снижение задержки при сравнимом качестве по сравнению с традиционными RAG-системами на более крупных моделях.
*   Вклад модулей CMPR и SCM является критическим: их удаление приводит к заметному падению качества, что подтверждает их роль в компенсации ограничений небольших генераторов.

## 📝 Критический анализ

```markdown
# LightRAG (2024)
---
[[paper]](https://arxiv.org/pdf/2402.13872)<br>LightRAG = Lightweight Retrieval-Augmented Generation

LightRAG — это фреймворк, предложенный исследователями из Университета Цинхуа, который фокусируется на создании эффективных и экономичных систем Retrieval-Augmented Generation (RAG).

### Проблема

Традиционные системы RAG, такие как Vanilla RAG (2020), имеют высокие вычислительные затраты, задержки и сложность управления, что ограничивает их внедрение.

### Идея

LightRAG предлагает **легковесный и экономичный RAG-фреймворк** с оптимизацией компонентов и новыми механизмами для компенсации ограничений небольших моделей:
1. Компактные LLM как генераторы.
2. Легковесные ретриверы.
3. Стратегии **уточнения промптов** и **самокоррекции**.

### Постановка задачи

LightRAG улучшает эффективность, снижает стоимость и задержки RAG-систем, сохраняя качество генерации для задач, таких как Question Answering.

### Существующие альтернативы

На момент появления LightRAG существовали Vanilla RAG (2020), REALM (2020), RETRO (2021) и гибридные ретриверы, которые были ресурсоемкими и сложными в управлении.

### Архитектура

Архитектура LightRAG включает четыре модуля:

1. **Lightweight Generator:** Небольшая LLM (например, GPT-2 Small), дообученная для RAG-задач.
2. **Lightweight Retriever:** Быстрый лексический ретривер (например, BM25) или легковесный Dense Retriever.
3. **Cross-Modality Prompt Refinement (CMPR):** Оптимизирует извлеченные фрагменты для генерации.
4. **Self-Correction Mechanism (SCM):** Контролирует качество генерации, инициируя повторный ретривал или генерацию при необходимости.

<img src="img/img.png" width=500>

### Алгоритм обучения

1. **Генератор:** Дообучение на RAG-наборах данных.
2. **Ретривер:** Обучение с контрастивным обучением.
3. **CMPR:** Обучение на данных для извлечения ключевой информации.
4. **SCM:** Настройка порогов или обучение классификатора для оценки качества.

### Алгоритм инференса

1. **Исходный ретривал:** Поиск релевантных фрагментов.
2. **Уточнение промпта:** Создание оптимизированного контекста.
3. **Исходная генерация:** Генерация ответа.
4. **Самокоррекция:** Анализ и возможное повторение ретривала или генерации.
5. **Вывод:** Финальный ответ.

### Результаты

LightRAG показывает значительное снижение вычислительных затрат и задержек при сохранении качества. На бенчмарках, таких как Natural Questions, LightRAG на Llama-2-7B достигает производительности, сопоставимой с более крупными системами, обеспечивая 2-3-кратное снижение задержки.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации LightRAG на Python

# Импорт необходимых библиотек
from transformers import AutoModelForCausalLM, AutoTokenizer
from rank_bm25 import BM25Okapi
import numpy as np

# 1. Lightweight Generator: Используем небольшую языковую модель
generator_model_name = "gpt2"  # Можно заменить на другую легковесную модель
generator = AutoModelForCausalLM.from_pretrained(generator_model_name)
tokenizer = AutoTokenizer.from_pretrained(generator_model_name)

# 2. Lightweight Retriever: Используем BM25 для лексического поиска
documents = [
    "The capital of France is Paris.",
    "The Eiffel Tower is located in Paris.",
    "The Louvre is the world's largest art museum.",
    "The Seine river flows through Paris."
]
tokenized_docs = [doc.split(" ") for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

# 3. Cross-Modality Prompt Refinement (CMPR): Уточнение промпта
def refine_prompt(query, retrieved_docs):
    # Простой пример: конкатенация запроса и первых двух извлеченных документов
    refined_prompt = query + " " + " ".join(retrieved_docs[:2])
    return refined_prompt

# 4. Self-Correction Mechanism (SCM): Механизм самокоррекции
def self_correction(generated_text, threshold=0.5):
    # Простой пример: если длина текста меньше порога, повторяем генерацию
    if len(generated_text.split()) < threshold:
        return True
    return False

# Функция для выполнения инференса LightRAG
def light_rag_inference(query):
    # Шаг 1: Исходный ретривал
    tokenized_query = query.split(" ")
    scores = bm25.get_scores(tokenized_query)
    top_k_indices = np.argsort(scores)[::-1][:2]  # Извлекаем топ-2 документа
    retrieved_docs = [documents[i] for i in top_k_indices]

    # Шаг 2: Уточнение промпта
    refined_prompt = refine_prompt(query, retrieved_docs)

    # Шаг 3: Исходная генерация
    inputs = tokenizer.encode(refined_prompt, return_tensors="pt")
    outputs = generator.generate(inputs, max_length=50, num_return_sequences=1)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Шаг 4: Самокоррекция
    if self_correction(generated_text):
        # Повторяем генерацию с улучшенным промптом
        refined_prompt += " Please provide more details."
        inputs = tokenizer.encode(refined_prompt, return_tensors="pt")
        outputs = generator.generate(inputs, max_length=50, num_return_sequences=1)
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return generated_text

# Пример использования
query = "Tell me about Paris."
response = light_rag_inference(query)
print("Response:", response)
```

### Объяснение ключевых моментов:

1. **Lightweight Generator:** Используется небольшая языковая модель (например, GPT-2), что снижает вычислительные затраты и задержки.

2. **Lightweight Retriever:** Применяется BM25 для быстрого и экономичного извлечения релевантных документов.

3. **Cross-Modality Prompt Refinement (CMPR):** Уточнение промпта осуществляется путем конкатенации запроса и извлеченных документов, что помогает компенсировать ограничения небольших моделей.

4. **Self-Correction Mechanism (SCM):** Простой механизм самокоррекции проверяет длину сгенерированного текста и при необходимости повторяет генерацию с улучшенным промптом.

Этот пример иллюстрирует основные концепции LightRAG, такие как легковесность и эффективность, а также использование интеллектуальных механизмов для улучшения качества генерации.